# INF01017 - Aprendizado de Máquina (UFRGS)
## **Prática: Otimização de Hiperparâmetros e Generalização**

**Professora:** Profa. Mariana Recamonde-Mendoza  
**Instituição:** Universidade Federal do Rio Grande do Sul – UFRGS

---

### Objetivo da atividade

Nesta prática vamos investigar, de forma experimental, **por que e como otimizar hiperparâmetros**.

Vamos comparar três famílias de modelos:

- **Regressão Logística** — modelo linear;
- **Árvore de Decisão** — modelo não linear baseado em particionamento;
- **MLP (rede neural)** — modelo não linear com parâmetros aprendidos por otimização.

Ao longo da atividade, vamos discutir:

1. parâmetros × hiperparâmetros;
2. capacidade do modelo, **underfitting** e **overfitting**;
3. o papel da regularização;
4. **holdout** e **nested cross-validation**;
5. **Grid Search × Randomized Search × Optuna**;
6. custo computacional × qualidade da busca;
7. por que o conjunto de teste deve permanecer intocado;
8. como transformar uma busca de hiperparâmetros em um experimento reprodutível.

> **Mensagem central:** otimizar hiperparâmetros não é apenas "testar vários valores". É uma etapa de seleção de modelos que precisa ser conduzida com bastante cuidado e atenção, e sem contaminar a avaliação final.

## 1. Preparação do ambiente

Vamos utilizar o **Adult Dataset**, o mesmo conjunto de dados explorado na atividade anterior.

O problema é uma classificação binária: prever se a renda anual é superior a US$ 50 mil.

A comparação será feita com o mesmo conjunto de dados e o mesmo pipeline de pré-processamento, para que a diferença observada seja atribuída principalmente ao **modelo e à sua configuração de hiperparâmetros**.

In [ ]:
# ============================================================
# 1. Preparação do ambiente
# ============================================================

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
    cross_validate,
    validation_curve,
    learning_curve,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
)

from scipy.stats import loguniform, randint, uniform

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Ambiente preparado.")

## 2. Carregamento e preparação do Adult Dataset

Vamos manter a mesma lógica da atividade anterior: transformar o alvo em uma variável binária.
Manteremos os registros com valores ausentes, os quais serão tratados pelo pipeline.

### Uma decisão importante

O **teste será separado antes da otimização**.

Depois disso:

```text
                    DATASET
                       │
             ┌─────────┴─────────┐
             │                   │
          TREINO               TESTE
       (otimização)         (avaliação final)
             │
      Nested Cross-Validation
             │
      ┌──────┴──────┐
      │             │
   INNER CV      OUTER CV
   seleciona     estima
   hiperparâmetros generalização
```

O teste **não participa da escolha dos hiperparâmetros**.

In [ ]:
# ============================================================
# 2. Adult Dataset
# ============================================================

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

print("Baixando Adult Dataset da UCI...")

df_adult = pd.read_csv(
    url,
    header=None,
    names=columns,
    skipinitialspace=True,
    na_values='?'
)

df_adult['target'] = (df_adult['income'] == '>50K').astype(int)

X_raw = df_adult.drop(columns=['income', 'target'])
y = df_adult['target'].values

print(f"Dimensões do dataset: {X_raw.shape}")
print(f"Proporção da classe positiva: {y.mean():.3f}")
print(f"Valores ausentes por coluna:\n{X_raw.isnull().sum()[X_raw.isnull().sum() > 0]}")

In [ ]:
# Tipos de atributos
num_cols = X_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_raw.select_dtypes(exclude=np.number).columns.tolist()

print("Atributos numéricos:", num_cols)
print("Atributos categóricos:", cat_cols)

## 3. Holdout: por que separar o teste antes de otimizar?

Vamos reservar **20% dos dados para o teste final**.

Os 80% restantes serão usados para:

- treinar os modelos;
- validar diferentes configurações;
- selecionar hiperparâmetros;
- comparar estratégias de busca;
- realizar a nested cross-validation.

### Um erro muito comum

Imagine que testamos 100 configurações e escolhemos aquela que obteve o melhor resultado no teste. Mesmo que nenhuma configuração tenha usado explicitamente os rótulos do teste para "treinar", o teste foi usado para **tomar uma decisão**. Nesse momento, ele deixou de ser um teste independente.

> O conjunto de teste deve responder apenas à pergunta: **"Como o procedimento completo se comporta em dados que nunca foram usados para tomar decisões?"**

In [ ]:
# ============================================================
# 3. Holdout estratificado
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Treino:", X_train.shape)
print("Teste :", X_test.shape)
print(f"Positivos no treino: {y_train.mean():.3f}")
print(f"Positivos no teste : {y_test.mean():.3f}")

# O conjunto de teste fica trancado até a avaliação do modelo final.
# Nenhuma decisão de modelagem será tomada com base nele.

## 4. Pipeline de pré-processamento

O pré-processamento também precisa respeitar a separação entre treino e validação.

Por isso, vamos colocar:

- imputação;
- normalização;
- one-hot encoding;

**dentro de um `Pipeline`**.

Isso é especialmente importante na validação cruzada: em cada fold, o pré-processamento deve ser ajustado apenas com os dados daquele fold de treinamento.

> Isso evita **data leakage**.

In [ ]:
# ============================================================
# 4. Pipeline de pré-processamento
# ============================================================

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

print("Pipeline de pré-processamento definido.")

## 5. Primeiro experimento: o que é um hiperparâmetro?

Antes de procurar "o melhor valor", vamos separar dois conceitos.

### Parâmetro

É aprendido durante o treinamento.

Exemplos:

- pesos da rede neural;
- coeficientes da regressão logística;
- estrutura concreta da árvore após o treinamento.

### Hiperparâmetro

É definido **antes do processo de treinamento** e controla como o modelo será aprendido.

Exemplos nesta atividade:

| Modelo | Hiperparâmetros |
|---|---|
| Regressão Logística | `C`, `penalty` |
| Árvore | `max_depth`, `min_samples_leaf`, `criterion` |
| MLP | `hidden_layer_sizes`, `alpha`, `learning_rate_init` |

A otimização de hiperparâmetros procura uma configuração que produza **boa generalização**, e não simplesmente o melhor ajuste ao conjunto de treinamento.

## 6. Underfitting × capacidade × overfitting

Uma das razões para otimizar hiperparâmetros é controlar a **capacidade efetiva do modelo**.

Um modelo pode ser:

- **simples demais : underfitting.*;
- adequado → boa generalização;
- **complexo demais → overfitting**.

### Como observar isso corretamente?

Nesta seção, usaremos **`validation_curve`** do scikit-learn.

Ela realiza **validação cruzada interna** para estimar o desempenho de validação sem tocar no conjunto de teste.

```python
train_scores, val_scores = validation_curve(
    estimator, X_train, y_train,
    param_name='model__max_depth',
    param_range=[1, 2, 3, 5, 10, 20],
    cv=cv, scoring='f1'
)
```

> **Por que não usar o conjunto de teste aqui?**  
> O teste só será usado na etapa de avaliação final (seção 23). Usá-lo agora — mesmo que apenas para "visualizar" — já constitui contaminação: as decisões visuais que tomamos ao interpretar os gráficos são decisões de modelagem.  
> A `validation_curve` demonstra o mesmo fenômeno de underfitting/overfitting sem comprometer a integridade experimental.

## 7. Capacidade dos modelos: a mesma pergunta para três algoritmos

| Modelo | Hiperparâmetro de capacidade |
|---|---|
| Regressão Logística | `C` |
| Árvore de Decisão | `max_depth` |
| MLP | `hidden_layer_sizes` (número total de neurônios) |

Em todos os casos usamos o mesmo pré-processamento e a mesma CV estratificada com 3 folds.

O eixo Y mostra a **média do F1** nos folds de treinamento e de validação.

A **faixa sombreada** representa ±1 desvio-padrão entre os folds — uma indicação da variabilidade do resultado.

In [ ]:
# ============================================================
# 6.1 Capacidade — Regressão Logística (via validation_curve)
# ============================================================

cv_cap = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

pipe_lr_cap = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

C_values = np.logspace(-3, 3, 7)

train_scores_lr, val_scores_lr = validation_curve(
    pipe_lr_cap,
    X_train, y_train,
    param_name='model__C',
    param_range=C_values,
    cv=cv_cap,
    scoring='f1',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.semilogx(C_values, train_scores_lr.mean(axis=1), 'o-', label='Treino (CV)')
ax.fill_between(
    C_values,
    train_scores_lr.mean(axis=1) - train_scores_lr.std(axis=1),
    train_scores_lr.mean(axis=1) + train_scores_lr.std(axis=1),
    alpha=0.15
)
ax.semilogx(C_values, val_scores_lr.mean(axis=1), 's--', label='Validação (CV)')
ax.fill_between(
    C_values,
    val_scores_lr.mean(axis=1) - val_scores_lr.std(axis=1),
    val_scores_lr.mean(axis=1) + val_scores_lr.std(axis=1),
    alpha=0.15
)

ax.axvline(C_values[np.argmax(val_scores_lr.mean(axis=1))],
           linestyle=':', color='gray', label='Melhor C (validação)')
ax.set_xlabel('C  (menor → maior regularização inversa)')
ax.set_ylabel('F1-score médio (k folds)')
ax.set_title('Regressão Logística: regularização × desempenho (validation_curve)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Nota: estas curvas são estimadas por CV interna — o conjunto de teste não foi usado.")

In [ ]:
# ============================================================
# 6.2 Capacidade — Árvore de Decisão (via validation_curve)
# ============================================================

pipe_tree_cap = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

depths = [1, 2, 3, 5, 8, 12, 20]

train_scores_tree, val_scores_tree = validation_curve(
    pipe_tree_cap,
    X_train, y_train,
    param_name='model__max_depth',
    param_range=depths,
    cv=cv_cap,
    scoring='f1',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(depths, train_scores_tree.mean(axis=1), 'o-', label='Treino (CV)')
ax.fill_between(
    depths,
    train_scores_tree.mean(axis=1) - train_scores_tree.std(axis=1),
    train_scores_tree.mean(axis=1) + train_scores_tree.std(axis=1),
    alpha=0.15
)
ax.plot(depths, val_scores_tree.mean(axis=1), 's--', label='Validação (CV)')
ax.fill_between(
    depths,
    val_scores_tree.mean(axis=1) - val_scores_tree.std(axis=1),
    val_scores_tree.mean(axis=1) + val_scores_tree.std(axis=1),
    alpha=0.15
)

best_depth = depths[np.argmax(val_scores_tree.mean(axis=1))]
ax.axvline(best_depth, linestyle=':', color='gray',
           label=f'Melhor max_depth={best_depth} (validação)')

ax.set_xlabel('max_depth')
ax.set_ylabel('F1-score médio (k folds)')
ax.set_title('Árvore de Decisão: profundidade × desempenho (validation_curve)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.3 Capacidade — MLP (via validation_curve)
# ============================================================
# Usamos o número total de neurônios como eixo contínuo.
# Todos os três modelos são avaliados no MESMO conjunto de treino.
# ============================================================

architectures_map = {
    8:  (8,),
    16: (16,),
    32: (32,),
    48: (16, 32),
    96: (32, 64),
    160: (32, 64, 64),
}
hidden_units_values = list(architectures_map.keys())
arch_values = list(architectures_map.values())

##early stopping usado para controlar tempo de execução
pipe_mlp_cap = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(max_iter=200, early_stopping=True,
                            n_iter_no_change=7,   # padrão é 10
                            random_state=RANDOM_STATE))
])



train_scores_mlp, val_scores_mlp = validation_curve(
    pipe_mlp_cap,
    X_train, y_train,
    param_name='model__hidden_layer_sizes',
    param_range=arch_values,
    cv=cv_cap,
    scoring='f1',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(arch_values))
labels = [str(a) for a in arch_values]

ax.plot(x, train_scores_mlp.mean(axis=1), 'o-', label='Treino (CV)')
ax.fill_between(
    x,
    train_scores_mlp.mean(axis=1) - train_scores_mlp.std(axis=1),
    train_scores_mlp.mean(axis=1) + train_scores_mlp.std(axis=1),
    alpha=0.15
)
ax.plot(x, val_scores_mlp.mean(axis=1), 's--', label='Validação (CV)')
ax.fill_between(
    x,
    val_scores_mlp.mean(axis=1) - val_scores_mlp.std(axis=1),
    val_scores_mlp.mean(axis=1) + val_scores_mlp.std(axis=1),
    alpha=0.15
)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20)
ax.set_xlabel('Arquitetura (camadas ocultas)')
ax.set_ylabel('F1-score médio (k folds)')
ax.set_title('MLP: capacidade da arquitetura × desempenho (validation_curve)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### O que esses três experimentos têm em comum?

A pergunta é sempre a mesma:

> **O que acontece quando aumentamos ou reduzimos a capacidade do modelo?**

Mas a forma de controlar a capacidade muda:

- na **Regressão Logística**, `C` altera a intensidade da regularização;
- na **Árvore**, `max_depth` limita a complexidade da estrutura;
- na **MLP**, a arquitetura determina a capacidade de representação.

Em geral, procuramos uma região em que:

- o modelo tenha capacidade suficiente para aprender;
- o desempenho em **validação** seja bom;
- a diferença entre treino e validação não seja excessiva.

**⚠️ Atenção:** todas essas curvas foram geradas com `validation_curve`, que usa CV interna.  
O conjunto de teste permanece intocado. Isso importa porque, se usássemos o teste para observar o fenômeno, já estaríamos tomando decisões com base nele — e ele deixaria de ser uma avaliação independente.

### Learning Curves: capacidade × tamanho do dataset

As curvas de validação mostram o efeito de **um hiperparâmetro**.

As **learning curves** mostram o efeito do **tamanho do conjunto de treinamento**.

Elas respondem a perguntas como:

- *Mais dados ajudariam esse modelo?*
- *O modelo está sofrendo de underfitting (alta variância de treino) ou overfitting (grande gap treino-validação)?*

Se as duas curvas convergem para um valor alto: bom ajuste.  
Se convergem para um valor baixo: underfitting — mais dados não resolve, é preciso aumentar a capacidade do modelo.  
Se há grande gap persistente: overfitting — regularização ou mais dados podem ajudar.

In [ ]:
# ============================================================
# 6.4 Learning Curves — Árvore de Decisão (exemplo)
# ============================================================

pipe_tree_lc = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
])

train_sizes, train_lc, val_lc = learning_curve(
    pipe_tree_lc,
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(train_sizes, train_lc.mean(axis=1), 'o-', label='Treino (CV)')
ax.fill_between(
    train_sizes,
    train_lc.mean(axis=1) - train_lc.std(axis=1),
    train_lc.mean(axis=1) + train_lc.std(axis=1),
    alpha=0.15
)
ax.plot(train_sizes, val_lc.mean(axis=1), 's--', label='Validação (CV)')
ax.fill_between(
    train_sizes,
    val_lc.mean(axis=1) - val_lc.std(axis=1),
    val_lc.mean(axis=1) + val_lc.std(axis=1),
    alpha=0.15
)

ax.set_xlabel('Tamanho do conjunto de treinamento')
ax.set_ylabel('F1-score médio (k folds)')
ax.set_title('Learning Curve — Árvore de Decisão (max_depth=5)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# print(
#     'Interpretação:
# '
#     '  - Gap grande treino-validacao: overfitting persiste mesmo com mais dados.
# '
#     '  - Curvas convergem para valor baixo: underfitting.
# '
#     '  - Curvas convergem para valor alto: bom ajuste.'
# )

## 8. O que significa otimizar hiperparâmetros?

Até agora, escolhemos alguns valores manualmente.

Mas, em um problema real, podemos ter:

- vários hiperparâmetros;
- muitos valores possíveis;
- interações entre hiperparâmetros.

Por exemplo, para uma árvore:

```python
max_depth = [2, 5, 10, 20]
min_samples_leaf = [1, 5, 10]
```

Temos:

$$4 \times 3 = 12$$

configurações possíveis.

A otimização de hiperparâmetros transforma a escolha manual em um **processo sistemático de busca**.

A pergunta passa a ser:

> **Dentro de um espaço de possibilidades, qual configuração apresenta melhor desempenho de validação?**

Mas surge um detalhe importante:

**não queremos escolher a configuração usando o conjunto de teste.**

Por isso, a busca será feita usando apenas os dados de desenvolvimento e **validação cruzada**.

### Um protocolo padrão para os três algoritmos

A partir daqui, vamos usar exatamente a mesma estrutura experimental para os três modelos.

```text
X_train
   │
   ▼
Pipeline
   │
   ├── pré-processamento
   │
   └── modelo
          │
          ▼
   Busca de hiperparâmetros
          │
          ▼
   Validação cruzada
          │
          ▼
   melhor configuração
```

A única coisa que muda entre os algoritmos é:

1. o estimador;
2. os hiperparâmetros que fazem sentido para aquele estimador.

Isso permite comparar **modelos**, sem misturar essa comparação com diferenças no protocolo experimental.

In [ ]:
# ============================================================
# 8.1 Pipelines padronizados
# ============================================================

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    'MLP': MLPClassifier(
       max_iter=200,
       early_stopping=True,
       n_iter_no_change=7,   # padrão é 10
       random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    for name, model in models.items()
}

for name, pipe in pipelines.items():
    print(f'\n{name}')
    print(pipe)

## 9. Espaços de busca

O espaço de busca representa uma **hipótese sobre onde procurar**.

Não existe um espaço universalmente correto: ele deve ser coerente com o modelo e com o problema.

### Regressão Logística

`C` controla a regularização de forma inversa:

- `C` pequeno → regularização forte;
- `C` grande → regularização fraca.

### Árvore de Decisão

- `max_depth` → limita a profundidade;
- `min_samples_leaf` → controla o tamanho mínimo das folhas.

### MLP

- `hidden_layer_sizes` → arquitetura;
- `alpha` → regularização L2;
- `learning_rate_init` → taxa inicial de aprendizagem.

> O algoritmo de busca pode ser sofisticado, mas ele não consegue compensar um espaço de busca mal definido.

In [ ]:
# ============================================================
# 9.1 Espaços de busca
#
# IMPORTANTE: para que a comparação Grid × Random × Optuna
# seja justa, os três métodos explorarão O MESMO espaço.
# O Grid usa a versão discretizada; Random e Optuna podem
# usar distribuições contínuas dentro dos mesmos limites.
# ============================================================

# ── Grade compartilhada (discretização do espaço) ──────────
grid_spaces = {
    'Logistic Regression': {
        'model__C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'Decision Tree': {
        'model__max_depth':      [2, 4, 8, 12, 20, None],
        'model__min_samples_leaf': [1, 5, 20]
    },
    'MLP': {
        'model__hidden_layer_sizes': [(16,), (32,), (64,), (16, 32), (32, 64)],
        'model__alpha':              [1e-4, 1e-2, 1e-1],
        'model__learning_rate_init': [1e-3, 5e-3, 1e-2]
    }
}

# ── Espaço contínuo para Random/Optuna (MESMOS LIMITES) ────
# Limites derivados da grade acima; distribuições log-uniformes
# para hiperparâmetros que variam em ordens de magnitude.
continuous_spaces = {
    'Logistic Regression': {
        'model__C': loguniform(0.001, 100)
    },
    'Decision Tree': {
        'model__max_depth':        randint(2, 21),
        'model__min_samples_leaf': randint(1, 21)
    },
    'MLP': {
        'model__hidden_layer_sizes': [(16,), (32,), (64,), (16, 32), (32, 64)],
        'model__alpha':              loguniform(1e-4, 1e-1),
        'model__learning_rate_init': loguniform(1e-3, 1e-2)
    }
}

for name, space in grid_spaces.items():
    n = 1
    for values in space.values():
        n *= len(values)
    print(f'{name}: {n} configurações no Grid Search')

## 10. Três estratégias para explorar o espaço

Agora vamos comparar **estratégias de busca**, não algoritmos de classificação.

As três estratégias recebem essencialmente a mesma tarefa:

> **Encontrar boas configurações de hiperparâmetros usando um orçamento de avaliações.**

### 🔲 Grid Search

Testa todas as combinações especificadas.

**Ponto forte:** simples e determinístico.

**Limitação:** o número de combinações cresce exponencialmente com o número de hiperparâmetros (*maldição da dimensionalidade*). Além disso, é forçado a usar valores discretos.

### 🎲 Randomized Search

Sorteia configurações de acordo com distribuições ou listas definidas.

**Ponto forte:** podemos definir diretamente `n_iter` (orçamento explícito). Pode explorar espaços contínuos.

### 🧭 Optuna

Realiza uma busca adaptativa: os resultados dos trials anteriores orientam os próximos via amostragem por árvore de Parzen (TPE).

**Ponto forte:** pode concentrar avaliações em regiões promissoras, visualizar a trajetória e estimar importância dos hiperparâmetros.

> **A comparação só é justa se controlarmos o orçamento E o espaço de busca.**  
> Usaremos o mesmo espaço (seção 9) e ~20 avaliações para cada método.

## 11. Primeiro: Grid Search

O `GridSearchCV` combina:

- um pipeline;
- um espaço de hiperparâmetros;
- uma estratégia de validação cruzada;
- uma métrica de avaliação.

Para cada configuração:

1. divide os dados nos folds;
2. treina o modelo;
3. calcula o F1 de validação;
4. calcula a média entre os folds.

No final, seleciona a configuração com maior **F1 médio de validação**.

Importante: **o teste não participa dessa decisão.**

In [ ]:
# ============================================================
# 11.1 Grid Search — mesmo protocolo para os três modelos
# ============================================================

cv_search = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid_results_summary = []
grid_searches = {}

for name, pipe in pipelines.items():

    n_configs = 1
    for v in grid_spaces[name].values():
        n_configs *= len(v)
    n_fits = n_configs * cv_search.n_splits
    print(f"[{name}] {n_configs} configurações × {cv_search.n_splits} folds = {n_fits} fits")

    start = time.perf_counter()

    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid_spaces[name],
        scoring='f1',
        cv=cv_search,
        n_jobs=-1,
        return_train_score=True
    )

    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    grid_searches[name] = search

    grid_results_summary.append({
        'Modelo': name,
        'Configs': n_configs,
        'Melhor F1 CV': search.best_score_,
        'Tempo (s)': round(elapsed, 1),
        'Melhor configuração': search.best_params_
    })

df_grid = pd.DataFrame(grid_results_summary).sort_values('Melhor F1 CV', ascending=False)
df_grid

In [ ]:
grid_searches['MLP'].best_params_

### O que o Grid Search nos entregou?

Para cada algoritmo, temos:

- a melhor configuração encontrada;
- o F1 médio nos folds de validação;
- o custo computacional da busca.

Observe que **o melhor F1 de CV não é ainda uma estimativa independente do desempenho final**.

Nós usamos os dados de validação para escolher a configuração. Portanto, precisamos ter cuidado ao interpretar esse valor.

Antes de pensar na avaliação final, vamos entender outras formas de realizar a busca.

## 12. Randomized Search: mesmo problema, número máximo de configurações explícito


No Grid Search, o número de avaliações é determinado pelo tamanho da grade.

No Randomized Search, definimos explicitamente:

```python
n_iter = 20  # número de configurações avaliadas
```

Isso é especialmente útil quando o espaço é grande ou contínuo.

Para comparar Grid e Random de forma justa, mantemos:

- **mesmo modelo:** os três;
- **mesmo espaço de busca** (limites idênticos, seção 9);
- **mesmo conjunto de dados**;
- **mesma CV** (3 folds);
- **mesma métrica** (F1);
- **orçamento semelhante** (~20 avaliações por modelo).

A diferença está apenas no **modo de explorar o espaço**.

In [ ]:
# ============================================================
# 12.1 Randomized Search — mesmo protocolo para os três modelos
# ============================================================

N_ITER = 20   # orçamento explícito; próximo do Grid para LR e DT

random_results_summary = []
random_searches = {}

for name, pipe in pipelines.items():

    n_fits = N_ITER * cv_search.n_splits
    print(f"[{name}] {N_ITER} iters × {cv_search.n_splits} folds = {n_fits} fits")

    start = time.perf_counter()

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=continuous_spaces[name],
        n_iter=N_ITER,
        scoring='f1',
        cv=cv_search,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        return_train_score=True
    )

    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    random_searches[name] = search

    random_results_summary.append({
        'Modelo': name,
        'Iters': N_ITER,
        'Melhor F1 CV': search.best_score_,
        'Tempo (s)': round(elapsed, 1),
        'Melhor configuração': search.best_params_
    })

df_random = pd.DataFrame(random_results_summary).sort_values('Melhor F1 CV', ascending=False)
df_random

## 13. Optuna: uma busca adaptativa

Em cada *trial*:

1. o Optuna propõe uma configuração (guiada pelo histórico);
2. treinamos e avaliamos com CV;
3. registramos o resultado;
4. os resultados acumulados ajudam a orientar a busca (TPE - [veja tutorial do Optuna](https://hub.optuna.org/samplers/tpe_tutorial/$0)).

```text
Trial 1 → configuração → avaliação
                         ↓
Trial 2 → configuração → avaliação
                         ↓
                 histórico da busca (TPE)
                         ↓
Trial 3 → proposta orientada pelos resultados anteriores
```

O objetivo continua sendo o mesmo:

> **maximizar o F1 de validação dentro de um orçamento de trials.**

### Instalação

```bash
pip install optuna optuna-integration[sklearn] plotly
```

In [ ]:
!pip install optuna optuna-integration[sklearn] plotly

In [ ]:
# ============================================================
# 13.1 Optuna — preparação
# ============================================================

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print("Optuna disponível:", optuna.__version__)
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna não instalada. Execute: !pip install optuna")

In [ ]:
# ============================================================
# 13.2 Funções objetivo para cada modelo
#      (mesmo espaço que Grid/Random — limites idênticos)
# ============================================================

def make_objective(name, pipe_template, X, y, cv):
    """Retorna a função objetivo do Optuna para um dado modelo."""

    def objective(trial):
        if name == 'Logistic Regression':
            params = {
                'model__C': trial.suggest_float('C', 0.001, 100, log=True),
            }
        elif name == 'Decision Tree':
            params = {
                'model__max_depth': trial.suggest_int('max_depth', 2, 20),
                'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            }
        elif name == 'MLP':
            arch_name = trial.suggest_categorical(
                'architecture', ['(16,)', '(32,)', '(64,)', '(16, 32)', '(32, 64)']
            )
            arch_map = {
                '(16,)': (16,), '(32,)': (32,), '(64,)': (64,),
                '(16, 32)': (16, 32), '(32, 64)': (32, 64)
            }
            params = {
                'model__hidden_layer_sizes': arch_map[arch_name],
                'model__alpha': trial.suggest_float('alpha', 1e-4, 1e-1, log=True),
                'model__learning_rate_init': trial.suggest_float(
                    'learning_rate_init', 1e-3, 1e-2, log=True),
            }

        # Clona o pipeline e aplica os parâmetros
        import copy
        pipe = copy.deepcopy(pipe_template)
        pipe.set_params(**params)

        scores = cross_validate(pipe, X, y, cv=cv, scoring='f1', n_jobs=-1)
        return scores['test_score'].mean()

    return objective

In [ ]:
# ============================================================
# 13.3 Executando o Optuna para os três modelos
# ============================================================

N_TRIALS = 20   # mesmo orçamento que Random Search

optuna_results_summary = []
optuna_studies = {}

if OPTUNA_AVAILABLE:
    for name, pipe in pipelines.items():

        print(f"\n[{name}] {N_TRIALS} trials × {cv_search.n_splits} folds = {N_TRIALS * cv_search.n_splits} fits (máximo)")

        objective_fn = make_objective(name, pipe, X_train, y_train, cv_search)

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
        )

        start = time.perf_counter()
        study.optimize(objective_fn, n_trials=N_TRIALS)
        elapsed = time.perf_counter() - start

        optuna_studies[name] = study

        optuna_results_summary.append({
            'Modelo': name,
            'Trials': len(study.trials),
            'Melhor F1 CV': study.best_value,
            'Tempo (s)': round(elapsed, 1),
            'Melhor configuração': study.best_params
        })

        print(f"  Melhor F1 CV: {study.best_value:.4f} | {study.best_params}")

    df_optuna = pd.DataFrame(optuna_results_summary).sort_values('Melhor F1 CV', ascending=False)
    print("\n", df_optuna)

### Visualizando a trajetória do Optuna

Uma vantagem do Optuna sobre Grid e Randomized Search é a possibilidade de **visualizar como a busca evoluiu**.

Isso torna o comportamento adaptativo *visível*: nos primeiros trials, o Optuna explora; nos trials seguintes, concentra a busca em regiões promissoras.

### Histórico de otimização

Mostra o valor do melhor trial acumulado a cada novo trial.

In [ ]:
# ============================================================
# 13.4a Histórico de otimização — MLP (exemplo)
# ============================================================

if OPTUNA_AVAILABLE:
    study_mlp = optuna_studies['MLP']
    trials_df = study_mlp.trials_dataframe()

    # Melhor valor acumulado até cada trial
    best_so_far = trials_df['value'].cummax()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Esquerda: todos os trials + envelope do melhor acumulado ---
    ax = axes[0]
    ax.scatter(trials_df.index, trials_df['value'],
               alpha=0.6, s=40, label='Trial individual')
    ax.plot(trials_df.index, best_so_far, 'r-', linewidth=2,
            label='Melhor acumulado')
    ax.set_xlabel('Trial')
    ax.set_ylabel('F1 de validação (CV)')
    ax.set_title('Optuna — MLP: histórico dos trials')
    ax.legend()
    ax.grid(alpha=0.25)

    # --- Direita: distribuição dos valores por arquitetura ---
    ax = axes[1]
    if 'params_architecture' in trials_df.columns:
        for arch in trials_df['params_architecture'].unique():
            mask = trials_df['params_architecture'] == arch
            ax.scatter(trials_df.loc[mask].index,
                       trials_df.loc[mask, 'value'],
                       alpha=0.7, s=50, label=arch)
        ax.set_xlabel('Trial')
        ax.set_ylabel('F1 de validação (CV)')
        ax.set_title('Optuna — MLP: trials por arquitetura')
        ax.legend(title='Arquitetura', fontsize=8)
        ax.grid(alpha=0.25)
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 13.4b Importância dos hiperparâmetros (via Optuna)
# ============================================================

if OPTUNA_AVAILABLE:
    try:
        importances = optuna.importance.get_param_importances(study_mlp)
        print("Importância dos hiperparâmetros — MLP:")
        for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
            bar = '█' * int(imp * 40)
            print(f"  {param:30s} {bar} {imp:.3f}")
    except Exception as e:
        print(f"Visualização de importância não disponível: {e}")

## 14. Grid × Random × Optuna: comparação controlada

Agora podemos fazer uma comparação conceitualmente justa.

Para que a comparação seja válida, mantivemos:

- **mesmo espaço de busca** (limites idênticos);
- **mesma tarefa:** maximizar F1;
- **mesma validação cruzada:** 3 folds;
- **orçamento semelhante:** ~20 avaliações;
- **mesmo conjunto de desenvolvimento.**

A pergunta é:

> **Como diferentes estratégias de busca utilizam um orçamento semelhante para explorar o mesmo espaço?**

Não estamos tentando provar que um método é universalmente melhor. Estamos observando como o **modo de explorar o espaço** influencia o resultado.

In [ ]:
# ============================================================
# 14.1 Comparação final dos otimizadores — todos os modelos
# ============================================================

comparison_rows = []

for name in pipelines.keys():
    gs = grid_searches[name]
    rs = random_searches[name]

    n_grid = len(gs.cv_results_['params'])

    row_grid = {
        'Modelo': name, 'Método': 'Grid Search',
        'Avaliações': n_grid,
        'Melhor F1 CV': gs.best_score_,
    }
    row_rand = {
        'Modelo': name, 'Método': 'Randomized Search',
        'Avaliações': N_ITER,
        'Melhor F1 CV': rs.best_score_,
    }
    comparison_rows.extend([row_grid, row_rand])

    if OPTUNA_AVAILABLE and name in optuna_studies:
        row_opt = {
            'Modelo': name, 'Método': 'Optuna',
            'Avaliações': len(optuna_studies[name].trials),
            'Melhor F1 CV': optuna_studies[name].best_value,
        }
        comparison_rows.append(row_opt)

df_comparison = pd.DataFrame(comparison_rows)

# Pivot para facilitar leitura
df_pivot = df_comparison.pivot(index='Modelo', columns='Método', values='Melhor F1 CV')
df_pivot = df_pivot.round(4)
print("Melhor F1 CV por modelo e método:")
print(df_pivot.to_string())

In [ ]:
# ============================================================
# 14.2 Visualização: comparação por modelo
# ============================================================

methods = df_comparison['Método'].unique()
model_names = list(pipelines.keys())
x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

for i, method in enumerate(methods):
    scores = [
        df_comparison.loc[
            (df_comparison['Modelo'] == m) & (df_comparison['Método'] == method),
            'Melhor F1 CV'
        ].values[0] if len(df_comparison.loc[
            (df_comparison['Modelo'] == m) & (df_comparison['Método'] == method)
        ]) > 0 else 0
        for m in model_names
    ]
    bars = ax.bar(x + i * width, scores, width, label=method, alpha=0.85)
    for bar, score in zip(bars, scores):
        if score > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                    f'{score:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Modelo')
ax.set_ylabel('Melhor F1 de validação (CV)')
ax.set_title('Comparação de estratégias de busca — mesmo espaço, mesmo orçamento')
ax.set_xticks(x + width)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0.5, 0.85)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Conclusão desta comparação

Não existe um "campeão" independente do contexto.

- **Grid Search** é simples e determinístico, mas limitado a espaços discretos e pequenos.
- **Randomized Search** permite explorar um espaço maior (inclusive contínuo) com orçamento controlado. É frequentemente tão bom quanto o Grid com fração do custo.
- **Optuna** pode usar os resultados anteriores para direcionar a busca — seu diferencial fica mais evidente em espaços grandes ou com muitas avaliações, e suas visualizações ajudam a entender o espaço.

A escolha depende de:

- tamanho e tipo do espaço de busca;
- orçamento computacional disponível;
- necessidade de reprodutibilidade;
- custo de cada avaliação individual.

## 15. O problema da seleção e da avaliação

Imagine que usamos a validação cruzada para testar várias configurações.

Escolhemos aquela que teve o melhor resultado.

Esse resultado é útil para **selecionar** o modelo.

Mas existe uma possível fonte de otimismo:

> estamos escolhendo a configuração justamente porque ela apresentou um bom resultado naquela amostra de validação.

Portanto, precisamos separar duas perguntas:

### Seleção

> **Qual configuração parece melhor?**

### Avaliação

> **Qual desempenho podemos esperar em dados não utilizados na seleção?**

A **Nested Cross-Validation** resolve essa separação criando dois níveis de validação cruzada.

## 16. Nested Cross-Validation: CV dentro de CV

A ideia é literalmente uma **CV dentro de outra CV**.

```text
DADOS DE DESENVOLVIMENTO
│
├── OUTER FOLD 1
│     │
│     ├── treino externo
│     │      │
│     │      └── INNER CV
│     │             ├── fold 1
│     │             ├── fold 2
│     │             └── fold 3
│     │
│     └── teste externo → avaliação
│
├── OUTER FOLD 2
│     │
│     ├── treino externo
│     │      └── INNER CV → seleção
│     └── teste externo → avaliação
│
└── OUTER FOLD 3
      │
      ├── treino externo
      │      └── INNER CV → seleção
      └── teste externo → avaliação
```

### CV interna → **seleção**

Dentro de cada fold externo, usamos a CV interna para escolher os hiperparâmetros.

### CV externa → **avaliação**

Depois de escolher os hiperparâmetros, avaliamos o modelo no fold externo, que **não participou da seleção**.

> **CV interna seleciona. CV externa avalia.**

### Por que não fazer simplesmente uma Grid Search e reportar o melhor F1?

Porque isso mistura duas funções:

```text
mesmos dados
    │
    ├── escolha da configuração
    │
    └── avaliação da configuração escolhida
```

Na Nested CV:

```text
dados do fold externo
        │
        ├── INNER CV
        │      └── escolhe hiperparâmetros
        │
        └── OUTER TEST
               └── avalia a configuração escolhida
```

O fold externo funciona como uma espécie de **dados não vistos durante o processo de seleção**.

Isso fornece uma estimativa mais adequada do desempenho do **procedimento completo de modelagem**.

## 17. Nested CV para os três modelos (com Grid Search interno)

Vamos aplicar a Nested CV a todos os três algoritmos usando Grid Search interno.

A estrutura é:

- **3 folds externos** (avaliação);
- **3 folds internos** (seleção);
- **Grid Search interno** com o mesmo espaço de antes.

O restante permanece igual: mesmos dados, mesma métrica, mesmo pipeline.

Isso torna a comparação entre os modelos defensável.

In [ ]:
# ============================================================
# 18.1 Nested CV genérica com Grid Search interno
# ============================================================

def nested_cv_grid(estimator, param_grid, X, y, outer_cv, inner_cv):
    """
    Nested cross-validation com Grid Search no loop interno.
    Retorna:
      - outer_scores: F1 em cada fold externo
      - best_params_per_fold: hiperparâmetros selecionados em cada fold
    """
    outer_scores = []
    best_params_per_fold = []

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):

        X_out_train = X.iloc[train_idx]
        X_out_test  = X.iloc[test_idx]
        y_out_train = y[train_idx]
        y_out_test  = y[test_idx]

        search = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring='f1',
            cv=inner_cv,
            n_jobs=-1
        )
        search.fit(X_out_train, y_out_train)

        y_pred = search.best_estimator_.predict(X_out_test)
        score  = f1_score(y_out_test, y_pred)

        outer_scores.append(score)
        best_params_per_fold.append(search.best_params_)

        print(f'  Fold {fold}: F1 externo = {score:.4f} | best_params = {search.best_params_}')

    return np.array(outer_scores), best_params_per_fold

In [ ]:
# ============================================================
# 18.2 Executando Nested CV para os três modelos
# ============================================================

outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

nested_results = {}

for name in pipelines:
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')

    start = time.perf_counter()

    scores, params = nested_cv_grid(
        pipelines[name],
        grid_spaces[name],
        X_train, y_train,
        outer_cv, inner_cv
    )

    nested_results[name] = {
        'scores': scores,
        'params': params,
        'time': time.perf_counter() - start
    }

In [ ]:
# ============================================================
# 18.3 Resumo da Nested CV
# ============================================================

nested_summary = []
for name, result in nested_results.items():
    nested_summary.append({
        'Modelo': name,
        'F1 médio (outer)': result['scores'].mean(),
        'F1 std (outer)':   result['scores'].std(),
        'Tempo (s)':         round(result['time'], 1)
    })

df_nested = pd.DataFrame(nested_summary).sort_values('F1 médio (outer)', ascending=False)
df_nested

## 18. Estabilidade dos hiperparâmetros entre folds

A Nested CV não apenas estima o desempenho — ela revela os hiperparâmetros escolhidos em cada fold externo.

Quando os três folds escolhem **valores diferentes**, isso é um sinal importante:

> O modelo é **sensível à partição dos dados**. A configuração ótima não é estável.

Quando todos os folds concordam, a seleção é mais robusta.

Essa análise é frequentemente ignorada, mas é uma das informações mais valiosas que a Nested CV fornece.

In [ ]:
# ============================================================
# 18.4 Análise de estabilidade dos hiperparâmetros selecionados
# ============================================================

print("Hiperparâmetros selecionados em cada fold externo:\n")

for name, result in nested_results.items():
    print(f"{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")

    params_df = pd.DataFrame(result['params'])
    scores_series = pd.Series(result['scores'], name='F1 externo')

    combined = pd.concat([params_df, scores_series], axis=1)
    combined.index = [f'Fold {i+1}' for i in range(len(combined))]
    print(combined.to_string())

    # Verificar variabilidade
    for col in params_df.columns:
        n_unique = params_df[col].nunique()
        if n_unique > 1:
            print(f"  ⚠️  '{col}' variou entre folds → instabilidade neste hiperparâmetro")
        else:
            print(f"  ✅  '{col}' = {params_df[col].iloc[0]} (estável)")
    print()

## 19. Nested CV com Optuna interno (MLP)

Para conectar todos os conceitos, vamos usar o **Optuna dentro da CV interna** da Nested CV.

A estrutura é:

```text
Pipeline → Espaço de busca → Optuna → CV interna → melhores hiperparâmetros → fold externo
```

Isso permite comparar: *a Nested CV com Optuna interno dá uma estimativa de generalização diferente da Nested CV com Grid interno?*

Vamos usar o mesmo espaço de busca da MLP para que a comparação seja justa.

In [ ]:
# ============================================================
# 18.5 Nested CV com Optuna interno — MLP
# ============================================================

def nested_cv_optuna_mlp(X, y, n_outer=3, n_inner=3, n_trials=20, random_state=42):
    import copy

    outer_cv = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state)
    outer_scores = []
    selected_params = []

    for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):

        X_out_train = X.iloc[train_idx]
        X_out_test  = X.iloc[test_idx]
        y_out_train = y[train_idx]
        y_out_test  = y[test_idx]

        inner_cv = StratifiedKFold(n_splits=n_inner, shuffle=True,
                                   random_state=random_state + outer_fold)

        arch_map = {
            '(16,)': (16,), '(32,)': (32,), '(64,)': (64,),
            '(16, 32)': (16, 32), '(32, 64)': (32, 64)
        }

        def objective(trial):
            arch_name = trial.suggest_categorical(
                'architecture', list(arch_map.keys()))
            alpha = trial.suggest_float('alpha', 1e-4, 1e-1, log=True)
            lr    = trial.suggest_float('learning_rate_init', 1e-3, 1e-2, log=True)

            pipe = Pipeline([
                ('preprocessor', preprocessor),
                ('model', MLPClassifier(
                    hidden_layer_sizes=arch_map[arch_name],
                    alpha=alpha,
                    learning_rate_init=lr,
                    max_iter=300,
                    early_stopping=True,
                    random_state=random_state
                ))
            ])
            scores = cross_validate(pipe, X_out_train, y_out_train,
                                    cv=inner_cv, scoring='f1', n_jobs=-1)
            return scores['test_score'].mean()

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=random_state + outer_fold)
        )
        study.optimize(objective, n_trials=n_trials)

        best_arch = arch_map[study.best_params['architecture']]
        final_pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', MLPClassifier(
                hidden_layer_sizes=best_arch,
                alpha=study.best_params['alpha'],
                learning_rate_init=study.best_params['learning_rate_init'],
                max_iter=300, early_stopping=True, random_state=random_state
            ))
        ])
        final_pipe.fit(X_out_train, y_out_train)
        y_pred = final_pipe.predict(X_out_test)
        outer_f1 = f1_score(y_out_test, y_pred)

        outer_scores.append(outer_f1)
        selected_params.append(study.best_params)
        print(f'  Fold externo {outer_fold}: F1 = {outer_f1:.4f} | {study.best_params}')

    return np.array(outer_scores), selected_params

if OPTUNA_AVAILABLE:
    print("Nested CV com Optuna interno — MLP")
    print("="*50)
    start = time.perf_counter()
    mlp_optuna_scores, mlp_optuna_params = nested_cv_optuna_mlp(
        X_train, y_train, n_outer=3, n_inner=3, n_trials=20,
        random_state=RANDOM_STATE
    )
    optuna_nested_time = time.perf_counter() - start

    print(f"\nF1 médio (outer): {mlp_optuna_scores.mean():.4f}")
    print(f"F1 std   (outer): {mlp_optuna_scores.std():.4f}")
    print(f"Tempo total:      {optuna_nested_time:.1f} s")

### Grid interno × Optuna interno: qual estimativa de generalização?

Agora podemos comparar as duas versões de Nested CV para a MLP:

- **Grid interno:** exaustivo, mas espaço discreto;
- **Optuna interno:** adaptativo, espaço contínuo, 20 trials.

A diferença nos F1 externos reflete a qualidade do **processo de seleção**, não apenas do modelo.

In [ ]:
# ============================================================
# 18.6 Comparação: Nested CV com Grid vs Optuna interno (MLP)
# ============================================================

if OPTUNA_AVAILABLE:
    grid_mlp_nested = nested_results['MLP']

    comparison_nested = pd.DataFrame([
        {
            'Método interno': 'Grid Search',
            'F1 médio (outer)': grid_mlp_nested['scores'].mean(),
            'F1 std (outer)':   grid_mlp_nested['scores'].std(),
            'Tempo (s)':         round(grid_mlp_nested['time'], 1)
        },
        {
            'Método interno': 'Optuna',
            'F1 médio (outer)': mlp_optuna_scores.mean(),
            'F1 std (outer)':   mlp_optuna_scores.std(),
            'Tempo (s)':         round(optuna_nested_time, 1)
        }
    ])
    print("Nested CV — MLP — comparação do método interno:")
    print(comparison_nested.to_string(index=False))

### O que a Nested CV está estimando?

O valor que queremos interpretar aqui é:

$$
\text{F1 médio dos folds externos}
$$

Ele representa o desempenho observado quando o **procedimento completo**:

1. recebe dados de treinamento;
2. escolhe hiperparâmetros usando a CV interna;
3. treina o modelo com a configuração escolhida;
4. é avaliado em dados externos à seleção.

Por isso, a Nested CV é particularmente útil quando estamos comparando procedimentos de modelagem e queremos uma estimativa menos otimista de generalização.

Também devemos observar o **desvio-padrão entre os folds**:

- média alta + baixa variabilidade → resultado mais consistente;
- média alta + alta variabilidade → resultado mais instável.



## 20. Depois da Nested CV: modelo final

A Nested CV nos ajuda a estimar o desempenho do procedimento.

Mas, ao final do desenvolvimento, ainda precisamos construir o **modelo que será avaliado no conjunto de teste**.

O fluxo agora é:

```text
Dados
 │
 ├── Desenvolvimento
 │      │
 │      └── Nested CV → escolha do procedimento
 │
 └── Teste
        ↑
        │
   avaliação final
```

Uma decisão precisa ser tomada sobre os hiperparâmetros finais.

Nesta atividade, vamos usar a configuração encontrada pela busca em todo o conjunto de desenvolvimento.

> Essa etapa é diferente da avaliação: depois de encerrada a seleção, podemos treinar o modelo final usando todos os dados de desenvolvimento.

In [ ]:
# ============================================================
# 20.1 Seleção da configuração final
# ============================================================

final_configs = {}

for name in pipelines:
    final_search = GridSearchCV(
        estimator=pipelines[name],
        param_grid=grid_spaces[name],
        scoring='f1',
        cv=cv_search,  # 5-fold sobre todo o treino
        n_jobs=-1
    )
    final_search.fit(X_train, y_train)
    final_configs[name] = final_search

    print(f'\n{name}')
    print('  Melhor configuração:', final_search.best_params_)
    print('  F1 CV (desenvolvimento):', f'{final_search.best_score_:.4f}')

## 21. Avaliação final no teste

🔓 **Somente agora o conjunto de teste entra no experimento.**

O teste responde:

> **"Depois de todo o processo de desenvolvimento, como o modelo se comporta em dados que permaneceram separados?"**

O teste não foi usado para:

- escolher hiperparâmetros;
- comparar configurações;
- escolher o método de busca;
- decidir o melhor modelo.

Por isso, sua avaliação fornece uma estimativa mais independente do desempenho final.

In [ ]:
# ============================================================
# 21.1 Avaliação final no conjunto de teste (primeira e única vez)
# ============================================================

final_test_results = []

for name, search in final_configs.items():

    final_model = search.best_estimator_
    final_model.fit(X_train, y_train)  # treina no conjunto de desenvolvimento completo

    y_pred = final_model.predict(X_test)
    y_prob = final_model.predict_proba(X_test)[:, 1]

    final_test_results.append({
        'Modelo': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
        'PR-AUC':    average_precision_score(y_test, y_prob),
    })

df_test = pd.DataFrame(final_test_results).sort_values('F1', ascending=False)
df_test.round(4)

In [ ]:
# ============================================================
# 21.2 Matrizes de confusão — todos os modelos
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name in zip(axes, pipelines.keys()):
    final_model = final_configs[name].best_estimator_
    y_pred = final_model.predict(X_test)

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['<=50K', '>50K'],
        ax=ax,
        colorbar=False
    )
    ax.set_title(name)

plt.suptitle('Matrizes de Confusão — Avaliação Final no Teste', y=1.02)
plt.tight_layout()
plt.show()

## 22. Nested CV × teste final

Agora podemos colocar lado a lado duas perspectivas.

### Nested CV

Estima o desempenho do **procedimento de seleção/modelagem** usando folds externos.

### Teste final

Avalia o **modelo final**, depois que todas as decisões de desenvolvimento foram encerradas.

Os valores não precisam ser idênticos.

Diferenças podem ocorrer por:

- variabilidade das amostras;
- diferenças entre os folds;
- instabilidade dos hiperparâmetros;
- diferenças entre o conjunto de desenvolvimento e o teste.

> Uma diferença entre Nested CV e teste não significa automaticamente que um dos resultados esteja "errado". Ela deve ser interpretada à luz da variabilidade e do protocolo experimental.

In [ ]:
# ============================================================
# 22.1 Comparação: Nested CV × teste
# ============================================================

comparison = df_nested[['Modelo', 'F1 médio (outer)', 'F1 std (outer)']].merge(
    df_test[['Modelo', 'F1']].rename(columns={'F1': 'F1 teste'}),
    on='Modelo'
)

comparison['Diferença (nested - teste)'] = (
    comparison['F1 médio (outer)'] - comparison['F1 teste']
).round(4)

print(comparison.to_string(index=False))

print("""
Interpretação:
  Nested CV   → estimativa de desempenho do procedimento (usada para comparação de modelos)
  Teste final → avaliação do modelo treinado em todo o conjunto de desenvolvimento
  Diferença   → quanto a estimativa de CV superestimou ou subestimou o desempenho real
""")

# 25. Exercício 1 — Heatmap: Árvore de Decisão

Investigue simultaneamente `max_depth` e `min_samples_leaf` usando um **heatmap de F1 de validação**.

### Perguntas

1. Qual combinação de `max_depth` × `min_samples_leaf` maximiza o F1 de validação?
2. Existe uma combinação com alto F1 de treino e baixo F1 de validação? O que isso indica?
3. Você escolheria necessariamente a configuração com maior F1 de treino? Por quê?

### Código-esqueleto

In [ ]:
# EXERCÍCIO 1 — complete e analise

from sklearn.model_selection import cross_val_score

depths = [2, 4, 8, 12, 20]
min_samples = [1, 5, 10, 20]

# Matrizes para armazenar resultados
train_f1_matrix = np.zeros((len(depths), len(min_samples)))
val_f1_matrix   = np.zeros((len(depths), len(min_samples)))

cv_ex2 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# TODO: preencha as matrizes com um loop duplo (i, depth) e (j, ms)
# Para cada combinação:
#   - use cross_validate com return_train_score=True
#   - armazene a média do treino em train_f1_matrix[i, j]
#   - armazene a média da validação em val_f1_matrix[i, j]

# for i, depth in enumerate(depths):
#     for j, ms in enumerate(min_samples):
#         pipe = Pipeline([...])
#         result = cross_validate(pipe, X_train, y_train, cv=cv_ex2,
#                                  scoring='f1', return_train_score=True, n_jobs=-1)
#         train_f1_matrix[i, j] = result['train_score'].mean()
#         val_f1_matrix[i, j]   = result['test_score'].mean()

# TODO: plote os dois heatmaps lado a lado (treino e validação)
# Dica: use plt.imshow() ou seaborn.heatmap()
# Dica: marque a célula com melhor F1 de validação

print("Implemente o heatmap acima.")

# TODO: responda às perguntas em uma célula de texto abaixo.

# 26. Exercício 2 — Interações entre hiperparâmetros da MLP

Investigue como `alpha` (regularização) e `hidden_layer_sizes` (capacidade) interagem.

### Perguntas

1. Uma rede maior sempre generaliza melhor?
2. O efeito de `alpha` depende da arquitetura escolhida?
3. Uma taxa de aprendizagem inadequada pode fazer uma boa arquitetura parecer ruim?
4. Por que a otimização de uma MLP tende a ser mais sensível do que a de uma árvore?

### Código-esqueleto

In [ ]:
# EXERCÍCIO 2 — complete e analise

architectures = [(16,), (32,), (64,), (16, 32), (32, 64)]
alphas = [1e-4, 1e-2, 1e-1]
lr_fixed = 1e-2   # fixe a taxa de aprendizagem

cv_ex3 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

results_ex3 = []

# TODO: loop sobre arquiteturas e alphas
# Para cada combinação, calcule o F1 médio de validação com cross_val_score
# Registre: {'architecture': ..., 'alpha': ..., 'val_f1': ...}

# for arch in architectures:
#     for alpha in alphas:
#         pipe = Pipeline([...])
#         scores = cross_val_score(pipe, X_train, y_train, cv=cv_ex3,
#                                   scoring='f1', n_jobs=-1)
#         results_ex3.append({'architecture': str(arch), 'alpha': alpha,
#                              'val_f1': scores.mean()})

# TODO: pivote os resultados e plote como heatmap ou barras agrupadas

# df_ex3 = pd.DataFrame(results_ex3)
# pivot = df_ex3.pivot(index='architecture', columns='alpha', values='val_f1')

print("Implemente a análise acima.")

# 27. Exercício 3 — Compare Grid, Random e Optuna

Escolha **um único modelo** (sugestão: MLP ou Árvore).

Mantenha constantes:

- dados (`X_train`, `y_train`);
- pipeline;
- métrica (F1);
- validação cruzada (3 folds);
- **espaço de busca** (use os limites da seção 9);
- **orçamento:** ~20 avaliações por método.

Registre:

| Método | Avaliações | Melhor F1 CV | Tempo (s) | Melhor configuração |
|---|---:|---:|---:|---|
| Grid | | | | |
| Random | | | | |
| Optuna | | | | |

### Pergunta principal

> **Qual estratégia encontrou uma solução mais promissora considerando também o custo computacional?**

Não responda apenas olhando para o maior F1.

### Perguntas adicionais

1. O Optuna convergiu para uma região específica do espaço? (analise o gráfico de histórico)
2. Qual hiperparâmetro teve maior importância segundo o Optuna?
3. Com orçamento muito pequeno (5 trials), o Optuna ainda seria vantajoso?

In [ ]:
# EXERCÍCIO 3 — complete a comparação

# Escolha o modelo:
MODEL_NAME = 'MLP'   # ou 'Decision Tree'
BUDGET = 20          # avaliações por método

pipe_ex4   = pipelines[MODEL_NAME]
grid_ex4   = grid_spaces[MODEL_NAME]
cont_ex4   = continuous_spaces[MODEL_NAME]
cv_ex4     = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

# TODO 1: Grid Search
# grid_result = GridSearchCV(pipe_ex4, grid_ex4, scoring='f1', cv=cv_ex4, n_jobs=-1)
# grid_result.fit(X_train, y_train)

# TODO 2: Randomized Search (n_iter=BUDGET)
# rand_result = RandomizedSearchCV(pipe_ex4, cont_ex4, n_iter=BUDGET,
#                                   scoring='f1', cv=cv_ex4,
#                                   random_state=RANDOM_STATE, n_jobs=-1)
# rand_result.fit(X_train, y_train)

# TODO 3: Optuna (n_trials=BUDGET)
# Reutilize make_objective() da seção 13 ou crie sua própria função objetivo
# study_ex4 = optuna.create_study(direction='maximize',
#              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# study_ex4.optimize(make_objective(MODEL_NAME, pipe_ex4, X_train, y_train, cv_ex4),
#                    n_trials=BUDGET)

# TODO 4: Monte a tabela comparativa e plote o histórico do Optuna

print("Implemente a comparação acima.")

# 28. Exercício 4 — Nested CV

Escolha um dos modelos e implemente uma Nested CV completa.

Compare:

1. F1 obtido por uma busca simples no conjunto de desenvolvimento;
2. F1 médio da Nested CV (folds externos);
3. F1 final no teste (seção 21).

### Perguntas

- Por que esses três valores podem ser diferentes?
- Qual deles foi usado para seleção?
- Qual deles foi obtido em dados externos à seleção?
- Qual deles corresponde à avaliação final?
- O desvio-padrão dos folds externos foi alto ou baixo? O que isso indica sobre a estabilidade?

### Código-esqueleto

In [ ]:
# EXERCÍCIO 4 — complete e analise

MODEL_NAME_EX5 = 'Decision Tree'   # escolha o modelo

# TODO 1: Busca simples (sem nested CV)
# simple_search = GridSearchCV(...)
# simple_search.fit(X_train, y_train)
# simple_f1_cv = simple_search.best_score_

# TODO 2: Nested CV usando nested_cv_grid() definida na seção 18
# outer_cv_ex5 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
# inner_cv_ex5 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
# nested_scores, nested_params = nested_cv_grid(
#     pipelines[MODEL_NAME_EX5], grid_spaces[MODEL_NAME_EX5],
#     X_train, y_train, outer_cv_ex5, inner_cv_ex5
# )

# TODO 3: Compare com o F1 no teste da seção 21
# test_f1 = df_test.loc[df_test['Modelo'] == MODEL_NAME_EX5, 'F1'].values[0]

# TODO 4: Tabela comparativa
# pd.DataFrame({
#     'Método':      ['Busca simples (CV)', 'Nested CV (outer)', 'Teste final'],
#     'F1':          [simple_f1_cv, nested_scores.mean(), test_f1],
#     'Std':         [None, nested_scores.std(), None],
#     'Usado para':  ['Seleção', 'Estimativa de generalização', 'Avaliação final']
# })

# TODO 5: Analise a estabilidade dos parâmetros selecionados em cada fold externo
# pd.DataFrame(nested_params)

print("Implemente a análise acima.")

# 29. Uma visão prática do custo computacional

Considere:

- 3 folds externos;
- 3 folds internos;
- 20 configurações;
- 3 modelos.

A busca interna pode envolver aproximadamente:

$$
3 \times 20 \times 3 = 180
$$

ajustes por modelo.

Para três modelos:

$$
180 \times 3 = 540
$$

ajustes.

E isso antes de considerar:

- diferentes seeds;
- múltiplas métricas;
- análise de threshold;
- repetições;
- pipelines mais complexos.

Por isso:

> **o orçamento computacional também faz parte do desenho experimental.**

Alternativas para reduzir o custo:
- usar `n_iter` pequeno no Random Search;
- usar pruning no Optuna (eliminar trials ruins antecipadamente);
- usar uma fração do treino para a busca, refinando depois no conjunto completo;
- usar cross-validação com menos folds no loop interno.

# 30. Checklist para um experimento de otimização

### Dados

- [ ] O teste foi separado antes da otimização?
- [ ] O pré-processamento está dentro do Pipeline?
- [ ] Existe risco de data leakage?

### Visualização de capacidade

- [ ] As curvas de underfitting/overfitting foram feitas com `validation_curve` (sem usar o teste)?
- [ ] As learning curves foram inspecionadas para entender o efeito do tamanho do dataset?

### Busca

- [ ] O espaço de hiperparâmetros é justificável?
- [ ] O orçamento está definido explicitamente?
- [ ] A métrica de seleção faz sentido para o problema?
- [ ] O espaço de busca é o mesmo para todos os métodos comparados?

### Validação

- [ ] A CV é adequada ao problema?
- [ ] Há estratificação quando necessário?
- [ ] A seleção e a avaliação estão separadas (Nested CV)?

### Estabilidade

- [ ] Os hiperparâmetros selecionados em cada fold externo foram analisados?
- [ ] A variabilidade entre folds foi observada e interpretada?

### Generalização

- [ ] Há evidência de underfitting?
- [ ] Há evidência de overfitting?

### Avaliação final

- [ ] O teste permaneceu intocado até a avaliação final?
- [ ] O resultado final é acompanhado de múltiplas métricas relevantes?
- [ ] A diferença entre Nested CV e teste foi interpretada?